# 03. 전처리와 품질 점검

raw 테이블을 바탕으로 stg 테이블을 만들고, 품질 점검을 수행한다.

In [22]:
from pathlib import Path
import sqlite3
import pandas as pd

## 1. 경로 설정 및 DB 연결

In [23]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_DIR = PROJECT_ROOT / "data"
DB_PATH = DATA_DIR / "seoul_bike.db"
SQL_DIR = PROJECT_ROOT / "sql"
REPORT_DIR = PROJECT_ROOT / "reports"

SQL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH:", DB_PATH)

conn = sqlite3.connect(DB_PATH)

PROJECT_ROOT: d:\seoul-bike-aarrr-analysis
DB_PATH: d:\seoul-bike-aarrr-analysis\data\seoul_bike.db


## 2. raw 테이블 확인

In [24]:
tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)
tables

,name
0,raw_ride_month
1,raw_signup_month
2,raw_station_snapshot
3,raw_station_usage_month
4,stg_month_station_snapshot_map
5,stg_ride_month
6,stg_signup_month
7,stg_station_snapshot
8,stg_station_usage_month


In [25]:
required = {"raw_signup_month", "raw_ride_month", "raw_station_snapshot", "raw_station_usage_month"}
existing = set(tables["name"])
missing = required - existing
if missing:
    raise ValueError(f"필수 raw 테이블이 없습니다: {missing}")
print("필수 raw 테이블 확인 완료")

필수 raw 테이블 확인 완료


## 3. stg 테이블 생성

In [5]:
CREATE_STG_SQL = r"""-- 02_create_stg_tables.sql
-- SQLite 기준 staging 테이블 생성 스크립트

DROP TABLE IF EXISTS stg_signup_month;
DROP TABLE IF EXISTS stg_ride_month;
DROP TABLE IF EXISTS stg_station_snapshot;
DROP TABLE IF EXISTS stg_station_usage_month;
DROP TABLE IF EXISTS stg_month_station_snapshot_map;

CREATE TABLE stg_signup_month AS
SELECT
  source_file,
  CAST(signup_ym AS INTEGER) AS month_ym,
  substr(CAST(signup_ym AS TEXT),1,4)||'-'||substr(CAST(signup_ym AS TEXT),5,2)||'-01' AS month_date,
  NULLIF(TRIM(member_type_raw),'') AS member_type,
  CASE WHEN age_band_raw IS NULL OR UPPER(TRIM(age_band_raw)) IN ('','NAN','NULL','NONE') THEN 'UNKNOWN'
       ELSE TRIM(age_band_raw) END AS age_band,
  CASE WHEN UPPER(TRIM(gender_raw))='M' THEN 'M'
       WHEN UPPER(TRIM(gender_raw))='F' THEN 'F'
       ELSE 'UNKNOWN' END AS gender,
  CAST(signup_cnt_raw AS INTEGER) AS signup_cnt,
  loaded_at
FROM raw_signup_month;

CREATE TABLE stg_ride_month AS
SELECT
  source_file,
  CAST(ride_ym AS INTEGER) AS month_ym,
  substr(CAST(ride_ym AS TEXT),1,4)||'-'||substr(CAST(ride_ym AS TEXT),5,2)||'-01' AS month_date,
  CAST(station_id_raw AS INTEGER) AS station_id,
  TRIM(station_name_raw) AS station_name_raw,
  CASE WHEN instr(TRIM(station_name_raw),'.')>0
       THEN TRIM(substr(TRIM(station_name_raw), instr(TRIM(station_name_raw),'.')+1))
       ELSE TRIM(station_name_raw) END AS station_name,
  TRIM(pass_type_raw) AS pass_type,
  CASE WHEN TRIM(pass_type_raw)='정기권' THEN 'SUBSCRIPTION'
       ELSE 'NON_SUBSCRIPTION' END AS pass_group,
  CASE WHEN UPPER(TRIM(gender_raw))='M' THEN 'M'
       WHEN UPPER(TRIM(gender_raw))='F' THEN 'F'
       ELSE 'UNKNOWN' END AS gender,
  CASE WHEN age_band_raw IS NULL OR UPPER(TRIM(age_band_raw)) IN ('','NAN','NULL','NONE') THEN 'UNKNOWN'
       ELSE TRIM(age_band_raw) END AS age_band,
  CAST(ride_cnt_raw AS INTEGER) AS ride_cnt,
  CAST(REPLACE(NULLIF(TRIM(exercise_amt_raw),''),',','') AS REAL) AS exercise_amt,
  CAST(REPLACE(NULLIF(TRIM(carbon_amt_raw),''),',','') AS REAL) AS carbon_amt,
  CAST(REPLACE(NULLIF(TRIM(distance_m_raw),''),',','') AS REAL) AS distance_m,
  CAST(REPLACE(NULLIF(TRIM(duration_min_raw),''),',','') AS REAL) AS duration_min,
  CASE WHEN CAST(station_id_raw AS INTEGER)=9980 OR TRIM(station_name_raw) LIKE '%AS센터%' THEN 1 ELSE 0 END AS is_ops_excluded,
  loaded_at
FROM raw_ride_month;

CREATE TABLE stg_station_snapshot AS
WITH base AS (
  SELECT
    snapshot_ym,
    source_file,
    CAST(station_id_raw AS INTEGER) AS station_id,
    TRIM(station_name_raw) AS station_name,
    TRIM(district_raw) AS district,
    TRIM(address_raw) AS address,
    CAST(REPLACE(NULLIF(TRIM(lat_raw),''),',','') AS REAL) AS lat,
    CAST(REPLACE(NULLIF(TRIM(lon_raw),''),',','') AS REAL) AS lon,
    NULLIF(TRIM(installed_at_raw),'') AS installed_at_raw,
    CAST(REPLACE(NULLIF(TRIM(rack_lcd_raw),''),',','') AS INTEGER) AS rack_lcd,
    CAST(REPLACE(NULLIF(TRIM(rack_qr_raw),''),',','') AS INTEGER) AS rack_qr,
    TRIM(operation_type_raw) AS operation_type,
    loaded_at
  FROM raw_station_snapshot
)
SELECT
  snapshot_ym, source_file, station_id, station_name, district, address,
  lat, lon, installed_at_raw, rack_lcd, rack_qr, operation_type,
  CASE
    WHEN operation_type='QR' THEN COALESCE(rack_qr,0)
    WHEN operation_type='LCD' THEN COALESCE(rack_lcd,0)
    WHEN operation_type='LCD,QR' THEN COALESCE(rack_lcd,0)+COALESCE(rack_qr,0)
    ELSE COALESCE(rack_qr,rack_lcd,0)
  END AS rack_cnt,
  loaded_at
FROM base
WHERE station_id IS NOT NULL;

CREATE TABLE stg_station_usage_month AS
SELECT
  source_file,
  CAST(usage_ym AS INTEGER) AS month_ym,
  substr(CAST(usage_ym AS TEXT),1,4)||'-'||substr(CAST(usage_ym AS TEXT),5,2)||'-01' AS month_date,
  TRIM(district_raw) AS district,
  TRIM(station_name_raw) AS station_name_raw,
  CASE WHEN instr(TRIM(station_name_raw),'.')>0
       THEN CAST(substr(TRIM(station_name_raw),1,instr(TRIM(station_name_raw),'.')-1) AS INTEGER)
       ELSE NULL END AS station_id,
  CASE WHEN instr(TRIM(station_name_raw),'.')>0
       THEN TRIM(substr(TRIM(station_name_raw),instr(TRIM(station_name_raw),'.')+1))
       ELSE TRIM(station_name_raw) END AS station_name,
  CAST(checkout_cnt_raw AS INTEGER) AS checkout_cnt,
  CAST(return_cnt_raw AS INTEGER) AS return_cnt,
  loaded_at
FROM raw_station_usage_month;

CREATE TABLE stg_month_station_snapshot_map (
  month_ym INTEGER PRIMARY KEY,
  snapshot_ym INTEGER NOT NULL
);

INSERT INTO stg_month_station_snapshot_map (month_ym, snapshot_ym) VALUES
(202401,202406),(202402,202406),(202403,202406),(202404,202406),(202405,202406),(202406,202406),
(202407,202412),(202408,202412),(202409,202412),(202410,202412),(202411,202412),(202412,202412),
(202501,202506),(202502,202506),(202503,202506),(202504,202506),(202505,202506),(202506,202506),
(202507,202512),(202508,202512),(202509,202512),(202510,202512),(202511,202512),(202512,202512);

CREATE INDEX IF NOT EXISTS idx_stg_signup_month_keys ON stg_signup_month(month_ym, gender, age_band);
CREATE INDEX IF NOT EXISTS idx_stg_ride_month_keys ON stg_ride_month(month_ym, gender, age_band, pass_group);
CREATE INDEX IF NOT EXISTS idx_stg_ride_station ON stg_ride_month(month_ym, station_id);
CREATE INDEX IF NOT EXISTS idx_stg_station_snapshot_keys ON stg_station_snapshot(snapshot_ym, station_id);
CREATE INDEX IF NOT EXISTS idx_stg_station_usage_keys ON stg_station_usage_month(month_ym, station_id);
"""

(SQL_DIR / "02_create_stg_tables.sql").write_text(CREATE_STG_SQL, encoding="utf-8")
conn.executescript(CREATE_STG_SQL)
conn.commit()
print("stg 테이블 생성 완료")

stg 테이블 생성 완료


## 4. 행 수 확인

In [26]:
stg_tables = ["stg_signup_month", "stg_ride_month", "stg_station_snapshot", "stg_station_usage_month", "stg_month_station_snapshot_map"]
row_count_df = pd.DataFrame([
    {"table": t, "row_count": pd.read_sql_query(f"SELECT COUNT(*) AS c FROM {t}", conn).iloc[0,0]}
    for t in stg_tables
])
row_count_df

,table,row_count
0,stg_signup_month,385
1,stg_ride_month,2463808
2,stg_station_snapshot,11108
3,stg_station_usage_month,65761
4,stg_month_station_snapshot_map,24


## 5. 월별 커버리지 확인

In [27]:
coverage_sql = """
SELECT 'stg_signup_month' AS table_name, MIN(month_ym) AS min_ym, MAX(month_ym) AS max_ym, COUNT(DISTINCT month_ym) AS month_count FROM stg_signup_month
UNION ALL
SELECT 'stg_ride_month', MIN(month_ym), MAX(month_ym), COUNT(DISTINCT month_ym) FROM stg_ride_month
UNION ALL
SELECT 'stg_station_usage_month', MIN(month_ym), MAX(month_ym), COUNT(DISTINCT month_ym) FROM stg_station_usage_month
UNION ALL
SELECT 'stg_station_snapshot', MIN(snapshot_ym), MAX(snapshot_ym), COUNT(DISTINCT snapshot_ym) FROM stg_station_snapshot
"""
coverage_df = pd.read_sql_query(coverage_sql, conn)
coverage_df

,table_name,min_ym,max_ym,month_count
0,stg_signup_month,202401,202512,24
1,stg_ride_month,202401,202512,24
2,stg_station_usage_month,202401,202512,24
3,stg_station_snapshot,202406,202512,4


## 6. 성별/대여구분/연령대 표준화 결과

In [28]:
gender_check = pd.read_sql_query("""
SELECT 'signup' AS dataset, gender, COUNT(*) AS row_count FROM stg_signup_month GROUP BY gender
UNION ALL
SELECT 'ride' AS dataset, gender, COUNT(*) AS row_count FROM stg_ride_month GROUP BY gender
ORDER BY dataset, gender
""", conn)
gender_check

,dataset,gender,row_count
0,ride,F,794279
1,ride,M,900463
2,ride,UNKNOWN,769066
3,signup,F,192
4,signup,M,192
5,signup,UNKNOWN,1


In [29]:
pass_group_check = pd.read_sql_query("""
SELECT pass_group, COUNT(*) AS row_count, SUM(ride_cnt) AS ride_cnt_sum
FROM stg_ride_month
GROUP BY pass_group
ORDER BY pass_group
""", conn)
pass_group_check

,pass_group,row_count,ride_cnt_sum
0,NON_SUBSCRIPTION,1118291,13586034
1,SUBSCRIPTION,1345517,67636579


In [30]:
age_band_check = pd.read_sql_query("""
    SELECT 'signup' AS dataset, age_band, COUNT(*) AS row_count
    FROM stg_signup_month
    GROUP BY age_band

    UNION ALL

    SELECT 'ride' AS dataset, age_band, COUNT(*) AS row_count
    FROM stg_ride_month
    GROUP BY age_band

    ORDER BY dataset, age_band
    """,conn)
age_band_check

,dataset,age_band,row_count
0,ride,20대,388651
1,ride,30대,382583
2,ride,40대,362560
3,ride,50대,327347
4,ride,60대,235488
5,ride,70대이상,104814
6,ride,~10대,263820
7,ride,기타,398545
8,signup,20대,48
9,signup,30대,48


## 7. null 및 0값 점검

In [31]:
null_check_sql = """
SELECT 'stg_signup_month' AS table_name, 'signup_cnt' AS column_name, SUM(CASE WHEN signup_cnt IS NULL THEN 1 ELSE 0 END) AS null_count FROM stg_signup_month
UNION ALL SELECT 'stg_ride_month', 'station_id', SUM(CASE WHEN station_id IS NULL THEN 1 ELSE 0 END) FROM stg_ride_month
UNION ALL SELECT 'stg_ride_month', 'ride_cnt', SUM(CASE WHEN ride_cnt IS NULL THEN 1 ELSE 0 END) FROM stg_ride_month
UNION ALL SELECT 'stg_station_snapshot', 'station_id', SUM(CASE WHEN station_id IS NULL THEN 1 ELSE 0 END) FROM stg_station_snapshot
UNION ALL SELECT 'stg_station_snapshot', 'rack_cnt', SUM(CASE WHEN rack_cnt IS NULL THEN 1 ELSE 0 END) FROM stg_station_snapshot
"""
null_check_df = pd.read_sql_query(null_check_sql, conn)
null_check_df

,table_name,column_name,null_count
0,stg_signup_month,signup_cnt,0
1,stg_ride_month,station_id,0
2,stg_ride_month,ride_cnt,0
3,stg_station_snapshot,station_id,0
4,stg_station_snapshot,rack_cnt,0


In [32]:
zero_check_sql = """
SELECT 'stg_signup_month' AS table_name, 'signup_cnt <= 0' AS check_item, COUNT(*) AS issue_count FROM stg_signup_month WHERE signup_cnt <= 0
UNION ALL SELECT 'stg_ride_month', 'ride_cnt <= 0', COUNT(*) FROM stg_ride_month WHERE ride_cnt <= 0
UNION ALL SELECT 'stg_station_snapshot', 'rack_cnt <= 0', COUNT(*) FROM stg_station_snapshot WHERE rack_cnt <= 0
UNION ALL SELECT 'stg_station_usage_month', 'checkout_cnt < 0', COUNT(*) FROM stg_station_usage_month WHERE checkout_cnt < 0
UNION ALL SELECT 'stg_station_usage_month', 'return_cnt < 0', COUNT(*) FROM stg_station_usage_month WHERE return_cnt < 0
"""
zero_check_df = pd.read_sql_query(zero_check_sql, conn)
zero_check_df

,table_name,check_item,issue_count
0,stg_signup_month,signup_cnt <= 0,0
1,stg_ride_month,ride_cnt <= 0,0
2,stg_station_snapshot,rack_cnt <= 0,0
3,stg_station_usage_month,checkout_cnt < 0,0
4,stg_station_usage_month,return_cnt < 0,0


## 8. 조인 성공률 확인

In [33]:
join_success_sql = """
WITH ride_station AS (
  SELECT DISTINCT r.month_ym, m.snapshot_ym, r.station_id
  FROM stg_ride_month r
  JOIN stg_month_station_snapshot_map m ON r.month_ym = m.month_ym
  WHERE r.station_id IS NOT NULL AND r.is_ops_excluded = 0
), joined AS (
  SELECT rs.month_ym, rs.snapshot_ym, rs.station_id, s.station_id AS matched_station_id
  FROM ride_station rs
  LEFT JOIN stg_station_snapshot s
    ON rs.snapshot_ym = s.snapshot_ym AND rs.station_id = s.station_id
)
SELECT month_ym, snapshot_ym, COUNT(*) AS ride_station_count,
       SUM(CASE WHEN matched_station_id IS NOT NULL THEN 1 ELSE 0 END) AS matched_station_count,
       ROUND(100.0 * SUM(CASE WHEN matched_station_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS join_success_rate_pct
FROM joined
GROUP BY month_ym, snapshot_ym
ORDER BY month_ym
"""
join_success_df = pd.read_sql_query(join_success_sql, conn)
join_success_df

,month_ym,snapshot_ym,ride_station_count,matched_station_count,join_success_rate_pct
0,202401,202406,2728,2708,99.27
1,202402,202406,2727,2707,99.27
2,202403,202406,2729,2711,99.34
3,202404,202406,2730,2719,99.60
4,202405,202406,2732,2724,99.71
5,202406,202406,2734,2729,99.82
6,202407,202412,2733,2691,98.46
7,202408,202412,2722,2696,99.04
8,202409,202412,2734,2713,99.23
9,202410,202412,2743,2723,99.27


In [34]:
station_usage_extract_df = pd.read_sql_query("""
SELECT month_ym, COUNT(*) AS row_count,
       SUM(CASE WHEN station_id IS NOT NULL THEN 1 ELSE 0 END) AS station_id_extracted_count,
       ROUND(100.0 * SUM(CASE WHEN station_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS extract_success_rate_pct
FROM stg_station_usage_month
GROUP BY month_ym
ORDER BY month_ym
""", conn)
station_usage_extract_df

,month_ym,row_count,station_id_extracted_count,extract_success_rate_pct
0,202401,2728,2727,99.96
1,202402,2728,2727,99.96
2,202403,2730,2729,99.96
3,202404,2730,2730,100.00
4,202405,2732,2732,100.00
5,202406,2734,2734,100.00
6,202407,2733,2733,100.00
7,202408,2722,2722,100.00
8,202409,2735,2734,99.96
9,202410,2745,2744,99.96


## 9. 점검 결과 저장

In [35]:
summary_md_path = REPORT_DIR / "quality_check_summary.md"

def df_to_md(df):
    """tabulate 없이 DataFrame을 Markdown 표 문자열로 변환한다."""
    if df is None or df.empty:
        return "_데이터 없음_"

    temp = df.copy().fillna("").astype(str)

    def clean_cell(x):
        x = str(x)
        x = x.replace("\n", "<br>")
        x = x.replace("|", "\\|")
        return x

    headers = [clean_cell(c) for c in temp.columns]
    rows = [[clean_cell(v) for v in row] for row in temp.values.tolist()]

    lines = []
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")

    for row in rows:
        lines.append("| " + " | ".join(row) + " |")

    return "\n".join(lines)

summary_md = f"""# 전처리 및 품질 점검 결과 요약

## 1. stg 테이블 행 수

{df_to_md(row_count_df)}

## 2. 월별 커버리지

{df_to_md(coverage_df)}

## 3. 성별 표준화 결과

{df_to_md(gender_check)}

## 4. 대여구분 표준화 결과

{df_to_md(pass_group_check)}

## 5. 연령대 표준화 결과

{df_to_md(age_band_check)}

## 6. Null 점검

{df_to_md(null_check_df)}

## 7. 0값/음수값 점검

{df_to_md(zero_check_df)}

## 8. 대여소 조인 성공률

{df_to_md(join_success_df)}

## 9. 대여소별 이용정보 대여소번호 추출률

{df_to_md(station_usage_extract_df)}

## 해석 메모

- `UNKNOWN` 성별은 공개데이터의 결측 또는 비표준 성별 값을 표준화한 결과이다.
- `SUBSCRIPTION`은 대여구분코드가 `정기권`인 경우이다.
- `NON_SUBSCRIPTION`은 정기권 외 이용권을 통합한 값이다.
- 조인 성공률이 낮은 월이 있다면 해당 월의 대여소번호와 스냅샷 기준월 매핑을 추가 점검해야 한다.
- `rack_cnt <= 0`인 대여소는 운영 효율 분석에서 제외하거나 별도 확인이 필요하다.
"""

summary_md_path.write_text(summary_md, encoding="utf-8")

print("Markdown 품질 점검 요약 저장 완료:")
print(summary_md_path)


Markdown 품질 점검 요약 저장 완료:
d:\seoul-bike-aarrr-analysis\reports\quality_check_summary.md


In [36]:
QUALITY_CHECK_SQL = r"""-- 03_quality_check_queries.sql
-- SQLite 기준 품질 점검 쿼리 모음

-- 1. stg 테이블 행 수
SELECT 'stg_signup_month' AS table_name, COUNT(*) AS row_count FROM stg_signup_month
UNION ALL SELECT 'stg_ride_month', COUNT(*) FROM stg_ride_month
UNION ALL SELECT 'stg_station_snapshot', COUNT(*) FROM stg_station_snapshot
UNION ALL SELECT 'stg_station_usage_month', COUNT(*) FROM stg_station_usage_month;

-- 2. 월별 커버리지
SELECT 'stg_signup_month' AS table_name, MIN(month_ym) AS min_ym, MAX(month_ym) AS max_ym, COUNT(DISTINCT month_ym) AS month_count FROM stg_signup_month
UNION ALL SELECT 'stg_ride_month', MIN(month_ym), MAX(month_ym), COUNT(DISTINCT month_ym) FROM stg_ride_month
UNION ALL SELECT 'stg_station_usage_month', MIN(month_ym), MAX(month_ym), COUNT(DISTINCT month_ym) FROM stg_station_usage_month;

-- 3. 스냅샷별 대여소 수
SELECT snapshot_ym, COUNT(*) AS station_count
FROM stg_station_snapshot
GROUP BY snapshot_ym
ORDER BY snapshot_ym;

-- 4. 성별 표준화 결과
SELECT 'signup' AS dataset, gender, COUNT(*) AS row_count FROM stg_signup_month GROUP BY gender
UNION ALL
SELECT 'ride' AS dataset, gender, COUNT(*) AS row_count FROM stg_ride_month GROUP BY gender;

-- 5. 대여구분 표준화 결과
SELECT pass_group, COUNT(*) AS row_count, SUM(ride_cnt) AS ride_cnt_sum
FROM stg_ride_month
GROUP BY pass_group;

-- 6. 연령대 표준화 결과
SELECT 'signup' AS dataset, age_band, COUNT(*) AS row_count
FROM stg_signup_month
GROUP BY age_band

UNION ALL

SELECT 'ride' AS dataset, age_band, COUNT(*) AS row_count
FROM stg_ride_month
GROUP BY age_band

ORDER BY dataset, age_band;

-- 7. null 점검
SELECT 'stg_signup_month' AS table_name, 'month_ym' AS column_name, SUM(CASE WHEN month_ym IS NULL THEN 1 ELSE 0 END) AS null_count FROM stg_signup_month
UNION ALL SELECT 'stg_signup_month', 'signup_cnt', SUM(CASE WHEN signup_cnt IS NULL THEN 1 ELSE 0 END) FROM stg_signup_month
UNION ALL SELECT 'stg_ride_month', 'station_id', SUM(CASE WHEN station_id IS NULL THEN 1 ELSE 0 END) FROM stg_ride_month
UNION ALL SELECT 'stg_ride_month', 'ride_cnt', SUM(CASE WHEN ride_cnt IS NULL THEN 1 ELSE 0 END) FROM stg_ride_month
UNION ALL SELECT 'stg_station_snapshot', 'station_id', SUM(CASE WHEN station_id IS NULL THEN 1 ELSE 0 END) FROM stg_station_snapshot
UNION ALL SELECT 'stg_station_snapshot', 'rack_cnt', SUM(CASE WHEN rack_cnt IS NULL THEN 1 ELSE 0 END) FROM stg_station_snapshot;

-- 8. 0/음수 점검
SELECT 'stg_signup_month' AS table_name, 'signup_cnt <= 0' AS check_item, COUNT(*) AS issue_count FROM stg_signup_month WHERE signup_cnt <= 0
UNION ALL SELECT 'stg_ride_month', 'ride_cnt <= 0', COUNT(*) FROM stg_ride_month WHERE ride_cnt <= 0
UNION ALL SELECT 'stg_station_snapshot', 'rack_cnt <= 0', COUNT(*) FROM stg_station_snapshot WHERE rack_cnt <= 0
UNION ALL SELECT 'stg_station_usage_month', 'checkout_cnt < 0', COUNT(*) FROM stg_station_usage_month WHERE checkout_cnt < 0
UNION ALL SELECT 'stg_station_usage_month', 'return_cnt < 0', COUNT(*) FROM stg_station_usage_month WHERE return_cnt < 0;

-- 9. 이용정보와 대여소 스냅샷 조인 성공률
WITH ride_station AS (
  SELECT DISTINCT r.month_ym, m.snapshot_ym, r.station_id
  FROM stg_ride_month r
  JOIN stg_month_station_snapshot_map m ON r.month_ym = m.month_ym
  WHERE r.station_id IS NOT NULL AND r.is_ops_excluded = 0
), joined AS (
  SELECT rs.month_ym, rs.snapshot_ym, rs.station_id, s.station_id AS matched_station_id
  FROM ride_station rs
  LEFT JOIN stg_station_snapshot s
    ON rs.snapshot_ym = s.snapshot_ym AND rs.station_id = s.station_id
)
SELECT
  month_ym,
  snapshot_ym,
  COUNT(*) AS ride_station_count,
  SUM(CASE WHEN matched_station_id IS NOT NULL THEN 1 ELSE 0 END) AS matched_station_count,
  ROUND(100.0 * SUM(CASE WHEN matched_station_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS join_success_rate_pct
FROM joined
GROUP BY month_ym, snapshot_ym
ORDER BY month_ym;

-- 10. 대여소별 이용정보 station_id 추출 성공률
SELECT
  month_ym,
  COUNT(*) AS row_count,
  SUM(CASE WHEN station_id IS NOT NULL THEN 1 ELSE 0 END) AS station_id_extracted_count,
  ROUND(100.0 * SUM(CASE WHEN station_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS extract_success_rate_pct
FROM stg_station_usage_month
GROUP BY month_ym
ORDER BY month_ym;
"""
(SQL_DIR / "03_quality_check_queries.sql").write_text(QUALITY_CHECK_SQL, encoding="utf-8")
conn.close()
print("완료")

완료
